# RAG sul fascicolo: vettoriale o ibrida?

Giorno 3, pomeriggio.

Costruite una ricerca sul fascicolo in tre modi, misurate quale trova il chunk giusto, poi fate rispondere un
modello con le fonti. Tutti i dati sono sintetici.

| Parte | Che cosa si fa |
|---|---|
| 1. L'archivio | Dividere i documenti in chunk, calcolare un vettore per chunk |
| 2. Tre ricerche | Ricerca esatta, vettoriale, ibrida: per ogni domanda, il chunk atteso è tra i primi? |
| 3. La risposta con le fonti | I chunk trovati vanno a un modello; si controlla ogni affermazione |
| 4. La tabella | I vostri numeri, da incollare nel foglio condiviso |

Le celle si eseguono in ordine, con Maiusc+Invio. Dove trovate **Da fare**, cambiate la cella e rieseguitela.

## 1. L'archivio

### 1.1 I chunk

Il fascicolo si taglia in pezzi piccoli, i chunk: sono loro i risultati di una ricerca, e ognuno tiene il
suo riferimento, perché senza riferimento non c'è citazione.

In [1]:
from rag import Archivio, chunk_del_corpus, estrai_parole_significative, tabella

chunk = chunk_del_corpus()
print(len(chunk), "chunk da", len({r.split(":")[0] for r, _ in chunk}), "documenti")
tabella(["Chunk", "Testo (inizio)"], [(riferimento, testo[:110]) for riferimento, testo in chunk[:4]])

146 chunk da 15 documenti


Chunk,Testo (inizio)
DOC-01:2,DOC-01 · Verbale di ricezione di denuncia-querela orale
DOC-01:3,"Commissariato di P.S. di Borgovecchio (località di fantasia) · 3 marzo 2025, ore 16:40"
DOC-01:4,"Avanti al sottoscritto ufficiale di P.G. è presente Giulia Neri, identificata a mezzo carta d'identità, che di"
DOC-01:5,"Io sottoscritta Giulia Neri, nata a Borgovecchio il 12 maggio 1971 e ivi residente, espongo quanto segue."


Un chunk è una riga di un documento. Il riferimento `DOC-05:8` vuol dire: riga 8 del file di DOC-05. È ciò che
permette di citare la fonte.

Nei verbali le domande e le risposte stanno su righe diverse. Un chunk fatto della sola risposta («Sì, ero in
cucina con lei») non dice a che cosa risponde: per questo ogni risposta viaggia con la sua domanda.

In [2]:
tabella(
    ["Chunk", "Testo"],
    [(r, testo) for r, testo in chunk if r in ("DOC-02:8", "DOC-05:6", "DOC-05:8", "DOC-52:6")],
)

Chunk,Testo
DOC-02:8,"Domanda: Era presente quando sua moglie ha ricevuto la telefonata? Risposta: Sì, ero in cucina con lei. Saranno state le dieci e mezza del 3 marzo."
DOC-05:6,"03/03/2025,Bonifico istantaneo,""-4.200,00"",IT87 D999 9999 9990 0000 0123 456"
DOC-05:8,"05/03/2025,Bonifico,""-1.500,00"",IT87 D999 9999 9990 0000 0123 465"
DOC-52:6,"17/02/2025,Bonifico istantaneo,""-3.100,00"",IT38 O999 9999 9990 0000 0987 654"


Guardate le righe 6 e 8 di DOC-05: due IBAN che differiscono per due cifre scambiate. E DOC-52: un altro estratto
conto, di un'altra persona, con altri IBAN. Nel corpus ci sono cinque documenti così, di un'altra frode: servono a
vedere se la ricerca li confonde con i nostri. Un chunk è una riga sola, e una riga non dice di quale caso parla.

### 1.2 Estrarre le parole significative

La ricerca per parole chiave lavora sulle parole di ogni chunk. Come si dividono conta:

In [3]:
print(estrai_parole_significative("Che cosa registra il sito alle 21:58:12?"))
print(estrai_parole_significative('15 198.51.100.23 - - [02/Mar/2025:21:58:12 +0100] "POST /verifica HTTP/1.1" 302'))

['registra', 'sito', '21:58:12']
['15', '198.51.100.23', '02', 'mar', '2025', '21:58:12', '0100', 'post', 'verifica', 'http', '1.1', '302']


L'orario resta intero, e le parole che non distinguono niente («che», «cosa», «il») spariscono.

### 1.3 L'archivio

Adesso l'archivio: le parole di ogni chunk per la ricerca per parole chiave, e un vettore per chunk per quella
vettoriale.

Più sotto, la cella `archivio = Archivio()` scarica il modello di embedding (1 GB) se non l'avete già
scaricato stamattina: serve la rete, una volta sola.

In [4]:
archivio = Archivio()
print(archivio.vettori.shape, "= chunk x lunghezza vettore")

(146, 768) = chunk x lunghezza vettore


## 2. Tre ricerche

Una domanda, tre modi di cercare: la ricerca esatta, per parole chiave; la vettoriale, per significato;
l'ibrida, le due insieme. Prima ciascuna da sola, poi il confronto. La funzione qui sotto mostra in una tabella i primi risultati di una o più
ricerche.

In [5]:
from IPython.display import display


def mostra_risultati(domanda, modi=("ricerca esatta", "vettoriale", "ibrida"), k=5):
    """Mostra i primi k chunk trovati per la domanda, per ciascuna ricerca in `modi`.

    Ogni riga della tabella è un chunk: la ricerca che l'ha trovato, il posto, il riferimento (documento e riga) e
    l'inizio del testo.
    """
    print(domanda)
    righe = []
    for modo in modi:
        for posto, (riferimento, testo) in enumerate(archivio.cerca(domanda, modo, k), 1):
            righe.append((modo, posto, riferimento, testo[:95]))
    display(tabella(["Ricerca", "Posto", "Chunk", "Testo (inizio)"], righe))

### 2.1 Ricerca esatta (parole chiave, BM25)

Ogni chunk riceve un punteggio per le parole che ha in comune con la domanda. Una parola rara nel fascicolo, come
un orario o un IBAN, pesa molto; una parola che compare ovunque pesa poco. Un chunk lungo non vince solo perché è
lungo. Trova ciò che è scritto esattamente; non sa che «coniuge» e «moglie» sono la stessa persona.

In [6]:
mostra_risultati("Che cosa risulta dal telefono alle 10:41?", modi=["ricerca esatta"])

Che cosa risulta dal telefono alle 10:41?


Ricerca,Posto,Chunk,Testo (inizio)
ricerca esatta,1,DOC-04:11,03/03/2025 10:41 · in uscita · +39 000 000 0101 · durata 0 secondi (chiamata non riuscita)
ricerca esatta,2,DOC-02:16,Domanda: Ha visto l'operazione? Risposta: L'ho vista fare il bonifico dal telefono: 4.200 euro.
ricerca esatta,3,DOC-01:9,Seguendo le sue indicazioni ho disposto dal telefono un bonifico istantaneo di 4.200 euro sul c
ricerca esatta,4,DOC-02:14,Domanda: Sua moglie ha comunicato dei codici? Risposta: Le è arrivato un codice per messaggio.
ricerca esatta,5,DOC-04:2,DOC-04 · Estratto dal telefono di Giulia Neri: messaggi e chiamate del 3 marzo 2025


**Risposta.** `DOC-04:11` è al primo posto: «10:41» è nella domanda e nel chunk, ed è una parola rara, quindi
pesa più di tutte le altre.

### 2.2 Ricerca vettoriale

Il modello di embedding trasforma ogni chunk, e la domanda, in 768 numeri: un vettore. Testi che dicono la stessa
cosa hanno vettori vicini, anche con parole diverse, anche in un'altra lingua. La ricerca prende i chunk con il
vettore più vicino a quello della domanda (similarità del coseno).

In [7]:
mostra_risultati("Il coniuge era in casa durante la chiamata?", modi=["vettoriale"])

Il coniuge era in casa durante la chiamata?


Ricerca,Posto,Chunk,Testo (inizio)
vettoriale,1,DOC-02:8,"Domanda: Era presente quando sua moglie ha ricevuto la telefonata? Risposta: Sì, ero in cucina"
vettoriale,2,DOC-02:14,Domanda: Sua moglie ha comunicato dei codici? Risposta: Le è arrivato un codice per messaggio.
vettoriale,3,DOC-02:10,Domanda: Che cosa ha sentito? Risposta: Mia moglie ha messo il vivavoce per un momento. Parlava
vettoriale,4,DOC-02:18,Domanda: Sua moglie aveva ricevuto altre comunicazioni dalla banca nei giorni precedenti? Rispo
vettoriale,5,DOC-03:10,Domanda: Che cosa le ha riferito? Risposta: Che il giorno prima l'aveva chiamata una persona de


**Risposta.** `DOC-02:8` è al primo posto, anche se non ha nessuna parola in comune con la domanda: il verbale
dice «moglie» e «telefonata». Il significato è lo stesso, e i vettori sono vicini.

### 2.3 Ricerca ibrida

Le due classifiche si fondono: ogni chunk prende 1 / (60 + posto) da ciascuna ricerca, e i due punti si sommano
(reciprocal rank fusion). Un chunk in alto in tutte e due vince; uno in alto in una sola può restare sopra. Adesso
le tre ricerche insieme, sulle stesse due domande.

In [8]:
mostra_risultati("Che cosa risulta dal telefono alle 10:41?")

Che cosa risulta dal telefono alle 10:41?


Ricerca,Posto,Chunk,Testo (inizio)
ricerca esatta,1,DOC-04:11,03/03/2025 10:41 · in uscita · +39 000 000 0101 · durata 0 secondi (chiamata non riuscita)
ricerca esatta,2,DOC-02:16,Domanda: Ha visto l'operazione? Risposta: L'ho vista fare il bonifico dal telefono: 4.200 euro.
ricerca esatta,3,DOC-01:9,Seguendo le sue indicazioni ho disposto dal telefono un bonifico istantaneo di 4.200 euro sul c
ricerca esatta,4,DOC-02:14,Domanda: Sua moglie ha comunicato dei codici? Risposta: Le è arrivato un codice per messaggio.
ricerca esatta,5,DOC-04:2,DOC-04 · Estratto dal telefono di Giulia Neri: messaggi e chiamate del 3 marzo 2025
vettoriale,1,DOC-01:14,Allego la schermata dell'SMS delle 10:12.
vettoriale,2,DOC-51:13,Allego la schermata dell'SMS delle 09:47 e la ricevuta del bonifico.
vettoriale,3,DOC-01:13,Non ricordo l'ora della telefonata né quella del bonifico.
vettoriale,4,DOC-51:7,Alle 09:55 mi telefona una donna. Dice di chiamare dal servizio carte.
vettoriale,5,DOC-04:2,DOC-04 · Estratto dal telefono di Giulia Neri: messaggi e chiamate del 3 marzo 2025


La risposta sta in `DOC-04:11`. Chi la trova?

- **ricerca esatta**: sì, primo risultato (l'orario «10:41» è nel chunk)
- **vettoriale**: no, non è tra i primi cinque
- **ibrida**: sì, primo risultato, grazie alle parole

Adesso una domanda che non usa le parole del documento:

In [9]:
mostra_risultati("Il coniuge era in casa durante la chiamata?")

Il coniuge era in casa durante la chiamata?


Ricerca,Posto,Chunk,Testo (inizio)
ricerca esatta,1,DOC-01:10,Durante la telefonata mi è arrivato un codice di sicurezza: non ricordo se gli ho detto il codi
ricerca esatta,2,DOC-01:11,"Finita la chiamata ho richiamato il numero, che risultava inesistente, e ho capito di essere st"
ricerca esatta,3,DOC-51:11,Nel pomeriggio la banca mi conferma che nessun operatore mi aveva chiamata.
ricerca esatta,4,DOC-03:10,Domanda: Che cosa le ha riferito? Risposta: Che il giorno prima l'aveva chiamata una persona de
ricerca esatta,5,DOC-04:11,03/03/2025 10:41 · in uscita · +39 000 000 0101 · durata 0 secondi (chiamata non riuscita)
vettoriale,1,DOC-02:8,"Domanda: Era presente quando sua moglie ha ricevuto la telefonata? Risposta: Sì, ero in cucina"
vettoriale,2,DOC-02:14,Domanda: Sua moglie ha comunicato dei codici? Risposta: Le è arrivato un codice per messaggio.
vettoriale,3,DOC-02:10,Domanda: Che cosa ha sentito? Risposta: Mia moglie ha messo il vivavoce per un momento. Parlava
vettoriale,4,DOC-02:18,Domanda: Sua moglie aveva ricevuto altre comunicazioni dalla banca nei giorni precedenti? Rispo
vettoriale,5,DOC-03:10,Domanda: Che cosa le ha riferito? Risposta: Che il giorno prima l'aveva chiamata una persona de


Il verbale dice «moglie» e «telefonata», non «coniuge» e «chiamata»: la risposta è in `DOC-02:8`.

- **ricerca esatta**: no, nessuna parola della domanda è nel chunk
- **vettoriale**: sì, primo risultato (confronta il significato)
- **ibrida**: no, i chunk in cima nella ricerca esatta lo spingono fuori dai primi cinque

### 2.4 La vostra domanda

**Da fare.** Scrivete una vostra domanda sul caso e guardate che cosa trovano le tre ricerche.

Se non vi viene in mente, provate: `mostra_risultati("Quanto è durata la telefonata con il falso operatore?")`.

In [10]:
mostra_risultati("Quanto è durata la telefonata con il falso operatore?")  # la vostra domanda

Quanto è durata la telefonata con il falso operatore?


Ricerca,Posto,Chunk,Testo (inizio)
ricerca esatta,1,DOC-01:13,Non ricordo l'ora della telefonata né quella del bonifico.
ricerca esatta,2,DOC-01:7,Poco dopo mi ha chiamato un operatore. Diceva di essere della banca.
ricerca esatta,3,DOC-01:10,Durante la telefonata mi è arrivato un codice di sicurezza: non ricordo se gli ho detto il codi
ricerca esatta,4,DOC-51:11,Nel pomeriggio la banca mi conferma che nessun operatore mi aveva chiamata.
ricerca esatta,5,DOC-01:5,"Io sottoscritta Giulia Neri, nata a Borgovecchio il 12 maggio 1971 e ivi residente, espongo qua"
vettoriale,1,DOC-01:11,"Finita la chiamata ho richiamato il numero, che risultava inesistente, e ho capito di essere st"
vettoriale,2,DOC-01:7,Poco dopo mi ha chiamato un operatore. Diceva di essere della banca.
vettoriale,3,DOC-01:10,Durante la telefonata mi è arrivato un codice di sicurezza: non ricordo se gli ho detto il codi
vettoriale,4,DOC-03:10,Domanda: Che cosa le ha riferito? Risposta: Che il giorno prima l'aveva chiamata una persona de
vettoriale,5,DOC-51:7,Alle 09:55 mi telefona una donna. Dice di chiamare dal servizio carte.


**Risposta, per la domanda suggerita.** La risposta c'è, in `DOC-04:10`: chiamata in entrata dal numero del falso operatore, «durata 17 minuti». Nessuna delle tre ricerche la trova tra i primi cinque: la riga del registro chiamate non dice «telefonata» né «operatore». In cima esce invece la denuncia, dove la signora dice di non ricordare l'ora della telefonata. Un modello che riceve quei chunk rischia di rispondere che il fascicolo non lo dice, mentre lo dice: in un documento che la ricerca non ha portato su.

### 2.5 Il golden dataset

Due domande non sono una misura. Il golden dataset ne ha quindici, ciascuna con il chunk in cui sta la risposta:
sei con un identificativo esatto, sei parafrasi, due in cui due documenti si contraddicono, una a cui il fascicolo
non risponde.

In [11]:
from IPython.display import display
from prova_golden_dataset import MODI, valuta_ricerche


def misura_ricerche(k):
    """Misura le tre ricerche sulle quindici domande del golden dataset, con i primi k risultati.

    Mostra due tabelle: domanda per domanda (il chunk atteso è tra i primi k? sì o no) e i conteggi per tipo di
    domanda. Restituisce i conteggi, che servono alla parte 4.
    """
    esiti, conteggi = valuta_ricerche(archivio, k)
    print(f"Primi {k} risultati")
    display(
        tabella(
            ["n", "Tipo", *MODI, "Domanda"],
            [
                (e["n"], e["tipo"], *("sì" if e[m] else "no" for m in MODI), e["domanda"])
                for e in esiti
                if e["chunk_attesi"]
            ],
        )
    )
    display(
        tabella(
            ["Tipo", "Domande", *MODI],
            [(tipo, c["domande"], *(c[m] for m in MODI)) for tipo, c in conteggi.items()],
        )
    )
    return conteggi


conteggi_primi_5 = misura_ricerche(5)

Primi 5 risultati


n,Tipo,ricerca esatta,vettoriale,ibrida,Domanda
1,identificativo,sì,sì,sì,Quale movimento ha come destinatario l'IBAN IT87 D999 9999 9990 0000 0123 465?
2,identificativo,sì,no,sì,Che cosa risulta dal telefono alle 10:41?
3,identificativo,sì,sì,sì,In quale documento compare l'indirizzo 192.0.2.88?
4,identificativo,sì,no,sì,Che cosa è successo alle 22:40:05?
5,identificativo,sì,no,sì,Che cosa registra il sito alle 21:58:12?
6,identificativo,sì,no,sì,Quale codice contiene il messaggio delle 10:27?
7,parafrasi,no,sì,no,Il coniuge era in casa durante la chiamata?
8,parafrasi,no,no,no,La signora aveva capito che la posta elettronica e la chiamata erano collegate?
9,parafrasi,no,sì,sì,Con quale scusa è stata convinta a muovere i risparmi?
10,parafrasi,no,sì,no,"A che ora del giorno doveva essere spostato il denaro, secondo la chat?"


Tipo,Domande,ricerca esatta,vettoriale,ibrida
identificativo,6,6,2,6
parafrasi,6,1,4,2
contraddizione,2,1,1,2


Leggete la tabella per tipo di domanda, non il totale.

- Sugli identificativi chi vince? Guardate la domanda 2: perché la ricerca vettoriale non trova un orario?
- Sulle parafrasi chi vince? Guardate la domanda 10: la chat è in inglese, la domanda in italiano.
- Le contraddizioni chiedono due chunk insieme, uno per versione. Chi li riporta tutti e due?
- Le domande 8 e 12 non le trova nessuno. Aprite la 12 con `mostra_risultati(...)`: che cosa trova al suo posto?

**Da fare.** Rifate la tabella con i primi 3 risultati. Cambia qualcosa? Che cosa vi dice su dove sta il chunk
giusto, quando una ricerca lo trova?

In [12]:
conteggi_primi_3 = misura_ricerche(3)

Primi 3 risultati


n,Tipo,ricerca esatta,vettoriale,ibrida,Domanda
1,identificativo,sì,sì,sì,Quale movimento ha come destinatario l'IBAN IT87 D999 9999 9990 0000 0123 465?
2,identificativo,sì,no,sì,Che cosa risulta dal telefono alle 10:41?
3,identificativo,sì,sì,sì,In quale documento compare l'indirizzo 192.0.2.88?
4,identificativo,sì,no,sì,Che cosa è successo alle 22:40:05?
5,identificativo,sì,no,sì,Che cosa registra il sito alle 21:58:12?
6,identificativo,sì,no,sì,Quale codice contiene il messaggio delle 10:27?
7,parafrasi,no,sì,no,Il coniuge era in casa durante la chiamata?
8,parafrasi,no,no,no,La signora aveva capito che la posta elettronica e la chiamata erano collegate?
9,parafrasi,no,sì,sì,Con quale scusa è stata convinta a muovere i risparmi?
10,parafrasi,no,sì,no,"A che ora del giorno doveva essere spostato il denaro, secondo la chat?"


Tipo,Domande,ricerca esatta,vettoriale,ibrida
identificativo,6,6,2,6
parafrasi,6,1,4,2
contraddizione,2,1,1,2


**Risposte.**

La tabella, misurata su un Mac con processore Apple (su un altro portatile un sì può diventare un no). È uguale con i primi 5 e con i primi 3:

| Tipo | Domande | Ricerca esatta | Vettoriale | Ibrida |
|---|---|---|---|---|
| identificativo | 6 | 6 | 2 | 6 |
| parafrasi | 6 | 1 | 4 | 2 |
| contraddizione | 2 | 1 | 1 | 2 |
| totale | 14 | 8 | 7 | 10 |

- **Identificativi.** Vince la ricerca esatta, 6 su 6; la vettoriale ne trova 2. Domanda 2: un orario come «10:41» non ha un significato da confrontare, e la vettoriale porta su chunk con altri orari, come l'SMS delle 10:12.
- **Parafrasi.** Vince la vettoriale, 4 su 6. Domanda 10: la chat dice «Move the money to the safe account by noon». Tra italiano e inglese le parole non coincidono quasi mai, il significato sì: il modello di embedding è multilingue.
- **Contraddizioni.** Servono due chunk insieme, uno per versione. La ricerca esatta completa la 13, la vettoriale la 14: solo l'ibrida le completa tutte e due.
- **Domande 8 e 12.** La 12 chiede se la banca ha recuperato la prima somma. La risposta è in `DOC-10:6` («Non è revocabile; la richiesta di richiamo ha avuto esito negativo») e in `DOC-03:16`: nessuno dei due dice «recuperare» né «somma». Al loro posto tornano il richiamo del secondo bonifico (`DOC-10:9`) e l'altra frode (`DOC-54:7`, dove la somma «è rientrata»): un chunk che sembra rispondere, del caso sbagliato. La 8 chiede se la signora aveva capito il collegamento tra email e chiamata; la risposta è `DOC-11:7` («non aver collegato l'email alla telefonata»), con parole diverse e una negazione.
- **I primi 3.** Non cambia niente: quando una ricerca trova il chunk giusto, lo mette già tra i primi tre. Allargare a cinque non salva una ricerca che ha sbagliato strada.

In sintesi: nessuna ricerca basta da sola. L'ibrida ha il totale migliore (10 su 14) ed è l'unica che regge su tutti i tipi, ma sulle parafrasi perde contro la vettoriale.

## 3. La risposta con le fonti

I primi cinque chunk della ricerca ibrida vanno a un modello, con un'istruzione: rispondere solo con quei chunk,
citare la fonte di ogni affermazione, dire quando il fascicolo non basta.

Il modello è quello scritto in `LAB_MODEL` nel file `.env` di questa cartella. Se manca, o se la rete non va, il
notebook usa risposte già calcolate: il resto dell'esercizio non cambia.

### 3.1 Che cosa vede il modello

In [13]:
import csv

from risposte import QUI, costruisci_prompt, leggi_env, rispondi, risposta_precalcolata

leggi_env()
with (QUI / "golden_dataset.csv").open(encoding="utf-8") as file:
    golden_dataset = {d["n"]: d for d in csv.DictReader(file)}

print(costruisci_prompt(golden_dataset["1"]["domanda"], archivio.cerca(golden_dataset["1"]["domanda"], "ibrida", 5)))

Rispondi alla domanda usando solo i chunk qui sotto, che vengono da un fascicolo.
Ogni affermazione porta la sua fonte tra parentesi quadre, nella forma [DOC-05, riga 6].
Se due chunk dicono cose diverse, riportale tutte e due con le loro fonti e dì che non coincidono.
Se i chunk non bastano a rispondere, scrivi che il fascicolo non lo dice: non completare con ciò che sembra probabile.
Rispondi in italiano, in poche righe.

Chunk:
[DOC-05, riga 8] 05/03/2025,Bonifico,"-1.500,00",IT87 D999 9999 9990 0000 0123 465
[DOC-52, riga 8] 19/02/2025,Bonifico,"-1.150,00",IT11 J999 9999 9990 0000 0456 123
[DOC-05, riga 6] 03/03/2025,Bonifico istantaneo,"-4.200,00",IT87 D999 9999 9990 0000 0123 456
[DOC-52, riga 6] 17/02/2025,Bonifico istantaneo,"-3.100,00",IT38 O999 9999 9990 0000 0987 654
[DOC-05, riga 3] data,descrizione,importo,iban_destinatario

Domanda: Quale movimento ha come destinatario l'IBAN IT87 D999 9999 9990 0000 0123 465?


### 3.2 La risposta, e le fonti da controllare

Questo è tutto ciò che il modello vede. Adesso la risposta:

In [14]:
async def chiedi_al_modello(numero, tutto=False):
    """Fa rispondere il modello alla domanda `numero` del golden dataset, e la stampa.

    Con tutto=False il modello riceve i primi 5 chunk della ricerca ibrida (RAG); con tutto=True l'intero
    fascicolo, senza ricerca. Se il modello non risponde (chiave mancante, rete), stampa la risposta già calcolata
    e il motivo. In fondo stampa i chunk dati al modello e quelli attesi, per il confronto.
    """
    domanda = golden_dataset[numero]["domanda"]
    chunk_per_il_modello = archivio.chunk if tutto else archivio.cerca(domanda, "ibrida", 5)
    try:
        risposta, origine = await rispondi(domanda, chunk_per_il_modello), "Risposta del modello, adesso."
    except Exception as errore:  # in aula l'esercizio continua anche senza modello
        risposta, quando = risposta_precalcolata(numero, "tutto" if tutto else "rag")
        origine = f"Il modello non ha risposto: {str(errore)[:200]}\nQuesta è una risposta già pronta, {quando}."
    print(f"Domanda {numero}: {domanda}\n\n[{origine}]\n\n{risposta}\n")
    if not tutto:
        print("Chunk dati al modello:", ", ".join(r for r, _ in chunk_per_il_modello))
    print("Chunk attesi:", golden_dataset[numero]["chunk_attesi"] or "nessuno: il fascicolo non lo dice")


await chiedi_al_modello("1")

Domanda 1: Quale movimento ha come destinatario l'IBAN IT87 D999 9999 9990 0000 0123 465?

[Risposta del modello, adesso.]

Il movimento con il destinatario IT87 D999 9999 9990 0000 0123 465 risale al 05/03/2025 ed è un bonifico per importo -1.500,00 [DOC-05, riga 8].
Un altro movimento presente (riga 6) ha un IBAN quasi identico ma termina con il codice 456 [DOC-05, riga 6].

Chunk dati al modello: DOC-05:8, DOC-52:8, DOC-05:6, DOC-52:6, DOC-05:3
Chunk attesi: DOC-05:8


**Da fare.** Per ogni affermazione della risposta, aprite il chunk citato (la cella qui sotto) e controllate che
lo dica davvero. È la regola del corso: un risultato è un indizio, si controlla sulla fonte.

In [15]:
def apri_fonte(riferimento):
    """Stampa il testo intero di un chunk, dato il riferimento (per esempio "DOC-05:8"): così si controlla una citazione."""
    print(riferimento, "|", dict(archivio.chunk)[riferimento])


apri_fonte("DOC-05:8")

DOC-05:8 | 05/03/2025,Bonifico,"-1.500,00",IT87 D999 9999 9990 0000 0123 465


**Risposta.** `DOC-05:8` è il bonifico del 5 marzo, 1.500 euro, verso l'IBAN che termina con **465**: è il movimento giusto. La trappola è `DOC-05:6`, il bonifico del 3 marzo, con l'IBAN che termina con **456**: due cifre scambiate. Una risposta che cita la riga 6, o tratta i due IBAN come lo stesso conto, è sbagliata, e lo si vede solo aprendo il chunk e confrontando le cifre una per una. Nella risposta già calcolata il modello cita la riga giusta, ma aggiunge che «le fonti non coincidono completamente»: una confusione inutile, nata proprio dall'IBAN quasi uguale.

### 3.3 Una contraddizione

Una contraddizione: due documenti danno due importi diversi per lo stesso bonifico. Il modello li riporta tutti e due?
E gli importi che cita sono tutti del nostro caso? Aprite ogni fonte con `apri_fonte(...)` e guardate di chi parla il documento.

In [16]:
await chiedi_al_modello("13")

Domanda 13: Quanto è stato trasferito con il primo bonifico?

[Risposta del modello, adesso.]

Il fascicolo riporta dati contrastanti sull'importo del primo bonifico. [DOC-52, riga 6] indica un bonifico istantaneo dell'17/02/2025 per -3.100,00 euro; tuttavia le testimonie verbali non coincidono con questo importo: [DOC-03, riga 14] cita una somma di 2.400 euro e [DOC-02, riga 16] afferma di aver visto un bonifico di 4.200 euro. Non è quindi possibile confermare una cifra unica basandosi su tutti i documenti presenti.

Chunk dati al modello: DOC-03:14, DOC-02:16, DOC-01:13, DOC-52:6, DOC-52:8
Chunk attesi: DOC-03:14 & DOC-01:9|DOC-02:16|DOC-05:6|DOC-10:6


**Risposta.** Il nostro fascicolo ha due versioni del primo bonifico: **4.200 euro** (denuncia `DOC-01:9`, marito `DOC-02:16`, movimenti del conto `DOC-05:6`, banca `DOC-10:6`) e **2.400 euro** (l'impiegata della filiale, `DOC-03:14`, che riporta ciò che le ha detto la signora). Una buona risposta le riporta tutte e due, con la fonte di ciascuna; il peso sta con 4.200, confermato dai movimenti e dalla banca. Tra i chunk dati al modello ci sono però anche due righe di `DOC-52`, l'estratto conto dell'altra frode: 3.100 euro, 17/02/2025. Nella risposta già calcolata il modello apre proprio con quella riga e la presenta come «il primo bonifico». Aprite `apri_fonte("DOC-52:6")`: l'IBAN non è nessuno di quelli del nostro caso. Una fonte vera, del caso sbagliato.

### 3.4 Una domanda senza risposta

La domanda a cui il fascicolo non risponde. Che cosa fa il modello? Un orario, qui, sarebbe inventato: tra i chunk
non ce n'è nessuno. Guardate però quali documenti cita: se compare una data di febbraio, viene dall'altra frode. Una
fonte vera, del caso sbagliato.

In [17]:
await chiedi_al_modello("15")

Domanda 15: A che ora è stato eseguito il primo bonifico?

[Risposta del modello, adesso.]

Il fascicolo non lo dice: [DOC-01, riga 13] afferma esplicitamente «Non ricordo l'ora della telefonata né quella del bonifico». I restanti chunk ([DOC-52], [DOC-05]) riportano sole date e importi delle operazioni, senza specificare il preciso orario di esecuzione.

Chunk dati al modello: DOC-01:13, DOC-52:8, DOC-52:6, DOC-05:8, DOC-05:6
Chunk attesi: nessuno: il fascicolo non lo dice


**Risposta.** Il fascicolo non lo dice. La signora non ricorda l'ora (`DOC-01:13`) e la banca scrive che gli orari di esecuzione saranno trasmessi su provvedimento dell'Autorità Giudiziaria (`DOC-10:12`). Una risposta corretta dice che il dato manca. Si può accettare una finestra dedotta dalle fonti, purché sia presentata come deduzione: il bonifico è stato fatto durante la telefonata iniziata alle 10:19 (`DOC-04:10`, `DOC-01:9`). Un orario preciso è inventato. Nella risposta già calcolata il modello dice giustamente che l'ora non c'è, ma cita la data del 17/02/2025 da `DOC-52:6`: è il primo bonifico dell'altra frode. La fonte esiste, il caso è sbagliato.

### 3.5 Quando la ricerca non trova il chunk giusto

La domanda 12, una delle due che nessuna ricerca trova. Il modello riceve cinque chunk sbagliati: che cosa risponde?

In [18]:
await chiedi_al_modello("12")

Domanda 12: La banca è riuscita a recuperare la prima somma?

[Risposta del modello, adesso.]

Il fascicolo non lo dice; non vi sono informazioni nel testo sul recupero della prima somma, mentre si fa riferimento solo a un secondo bonifico. Per questo specifico trasferimento [DOC-54, riga 7] riporta che la somma è rientrata, ma [DOC-10, riga 9] afferma che l'esito non è ancora noto, creando una contraddizione sui dati disponibili.

Chunk dati al modello: DOC-03:10, DOC-54:7, DOC-10:9, DOC-06:16, DOC-04:6
Chunk attesi: DOC-10:6|DOC-03:16


**Risposta.** No: il primo bonifico è istantaneo, non è revocabile, e la richiesta di richiamo ha avuto esito negativo (`DOC-10:6`, confermato da `DOC-03:16`). Ma nessuno dei due chunk arriva al modello. Il modello riceve invece `DOC-10:9`, sul richiamo del *secondo* bonifico (esito non ancora noto), e `DOC-54:7`, dell'altra frode, dove la somma «è rientrata». Con questi chunk la risposta onesta è «il fascicolo, per quanto vedo, non lo dice»; quella pericolosa è «sì, la somma è rientrata», presa dal caso sbagliato. Nella risposta già calcolata il modello dice che non lo sa, ma nella spiegazione mescola i due casi. Il problema qui non è il modello: è la ricerca, che non gli ha dato il chunk giusto.

### 3.6 Tutto il fascicolo nel prompt

Il confronto: gli stessi quesiti con **tutto il fascicolo** nel prompt, senza ricerca. Con quindici documenti ci
sta. Risponde meglio? E con quarantamila documenti?

In [19]:
await chiedi_al_modello("12", tutto=True)

Domanda 12: La banca è riuscita a recuperare la prima somma?

[Il modello non ha risposto: il modello non ha risposto in 180 secondi
Questa è una risposta già pronta, calcolata il 2026-11-20 con ollama:qwen3.5-4b-16k.]

Per la prima somma di 4.200 euro (3 marzo), il documento DOC-10 afferma che non è stato revocabile e la richiesta di richiamo ha avuto un esito negativo [DOC-10, riga 6]. Di conseguenza, il fascicolo non indica che la banca sia riuscita a recuperare l'importo.

Chunk attesi: DOC-10:6|DOC-03:16


In [20]:
await chiedi_al_modello("13", tutto=True)

Domanda 13: Quanto è stato trasferito con il primo bonifico?

[Risposta del modello, adesso.]

Il fascicolo riferisce importo diverso per il "primo bonifico" a seconda della vittima coinvolta nel documento. Per la signora Giulia Neri, il primo bonifico del 3 marzo 2025 ha un importo di 4.200 euro [DOC-05, riga 6; DOC-10, riga 6]. Per la signora Marta Bassi, citata in documenti successivi (DOC-51/54), l'importo del primo bonifico è stato pari a 3.100 euro [DOC-54, riga 6].

Chunk attesi: DOC-03:14 & DOC-01:9|DOC-02:16|DOC-05:6|DOC-10:6


**Risposta.** Con tutto il fascicolo nel prompt il modello vede anche `DOC-10:6`. Nelle risposte già calcolate: sulla 12 risponde che la prima somma non è stata recuperata; sulla 13 separa i due casi (4.200 euro per Giulia Neri, 3.100 per l'altra vittima) invece di mescolarli. Però non riporta i 2.400 euro di `DOC-03:14`: anche con tutte le fonti davanti, la contraddizione va cercata da chi legge. Qui il prompt intero funziona perché quindici documenti, circa 16.000 caratteri, ci stanno. Con quarantamila documenti non ci stanno, e anche dove ci stessero il costo e il tempo di ogni domanda crescerebbero con il fascicolo: per questo serve la ricerca, e serve che trovi il chunk giusto.

## 4. La tabella

### 4.1 Le righe per il foglio condiviso

Selezionate la tabella qui sotto, copiatela e incollatela nel foglio condiviso: la guardiamo insieme alla fine.

In [21]:
# Una riga per tipo di domanda, con i primi 5 e con i primi 3 risultati: quante domande trova ciascuna ricerca.
tabella(
    ["Risultati", "Tipo di domanda", "Domande", *MODI],
    [
        (nome, tipo, valori["domande"], *(valori[m] for m in MODI))
        for nome, conteggi in (("primi 5", conteggi_primi_5), ("primi 3", conteggi_primi_3))
        for tipo, valori in conteggi.items()
    ],
)

Risultati,Tipo di domanda,Domande,ricerca esatta,vettoriale,ibrida
primi 5,identificativo,6,6,2,6
primi 5,parafrasi,6,1,4,2
primi 5,contraddizione,2,1,1,2
primi 3,identificativo,6,6,2,6
primi 3,parafrasi,6,1,4,2
primi 3,contraddizione,2,1,1,2


### 4.2 Discussione

Quattro domande per la discussione:

1. Per quale tipo di domanda usereste ciascuna ricerca?
2. Dove la ricerca ibrida ha fatto peggio della migliore delle altre due, e perché?
3. Quando il chunk giusto non è tra quelli trovati, che cosa ha fatto il modello?
4. Una citazione che esiste basta? Che cosa manca a un chunk per sapere di quale caso parla?

**Risposte.**

1. **Quale ricerca per quale domanda.** Parole chiave per ciò che si scrive esattamente: IBAN, orari, indirizzi IP, codici. Vettoriale quando si cerca un fatto con parole proprie, o in un'altra lingua. Ibrida quando non si sa in anticipo che domanda arriverà: non è la migliore su ogni tipo, ma non crolla su nessuno.
2. **Dove l'ibrida ha fatto peggio.** Sulle parafrasi: 2 su 6, contro 4 della vettoriale (domande 7 e 10). I chunk sbagliati che hanno qualche parola in comune con la domanda salgono nella classifica della ricerca esatta, e la fusione li tiene sopra il chunk giusto che la vettoriale aveva trovato.
3. **Che cosa ha fatto il modello senza il chunk giusto.** Dipende dal modello, e va letto sullo schermo. Nelle risposte già calcolate: sulla 15 ha detto che l'orario non c'è, ma ha preso una data dall'altra frode; sulla 12 ha detto che non lo sa, mescolando i due casi nella spiegazione. Un modello non può citare un chunk che non ha ricevuto.
4. **Una citazione che esiste basta?** No. Va aperta, e va guardato di chi parla il documento. A un chunk manca il contesto: di chi è il documento, di quale caso. Un sistema vero lo aggiunge a ogni chunk, per esempio con il titolo del documento e il numero del fascicolo, oppure filtra per fascicolo prima di cercare.